## Entity MiniLM recovery 

The actual `entity_inconsistency_plan` was indeed created in memory using `make_entity_inconsistency_plan(..., seed=42)`, but apparently it was never saved as a CSV.

What was frozen was `entity_inconsistency_affected_campaign_ids.csv`, containing the 30/90/180 affected IDs.

This means we shouldn't simply regenerate Entity and assume it's identical. However, we have a very solid way to recover the exact source of truth:
we reconstruct the defect using the same `seed=42`, temporarily reproduce the MiniLM embeddings, and compare them against the current official .npy files in `data/processed/`.

In [1]:
import importlib
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

import src.paths as paths
import src.defects as defects
import src.representations as representations

importlib.reload(paths)
importlib.reload(defects)
importlib.reload(representations)

from src.paths import get_project_paths
from src.defects import make_entity_inconsistency_plan,inject_entity_inconsistency
from src.representations import serialize_campaign_dataset,encode_serialized_campaigns

PATHS=get_project_paths()
DATA_DIR=PATHS["processed"]

In [2]:
# load formal clean data and split
clean_campaigns=pd.read_csv(
    DATA_DIR/"clean_structured_campaigns.csv",
    parse_dates=["timestamp"]
)

split_df=pd.read_csv(
    DATA_DIR/"structural_campaign_split.csv"
)

eval_ids=set(
    split_df.loc[
        split_df["split"]=="evaluation",
        "campaign_id"
    ]
)

clean_eval=(
    clean_campaigns[
        clean_campaigns["campaign_id"].isin(eval_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

print("Evaluation campaigns:",clean_eval["campaign_id"].nunique())
print("Messages:",len(clean_eval))

assert clean_eval["campaign_id"].nunique()==300

Evaluation campaigns: 300
Messages: 2090


In [3]:
# reconstruct original plan
entity_catalog=pd.read_csv(
    DATA_DIR/"entity_catalog.csv"
)

entity_plan=make_entity_inconsistency_plan(
    clean_eval,
    entity_catalog,
    seed=42
)

print(entity_plan.shape)
display(entity_plan.head())

(300, 7)


,campaign_id,entity_id,target_message_id,target_message_order,canonical_name,inconsistent_name,severity_rank
0,C0003,E0003,C0003_M03,3,Julia Collins,Julia Rivera,244
1,C0004,E0004,C0004_M05,5,Olivia Bailey,Olivia Medina,107
2,C0006,E0006,C0006_M05,5,Valeria Silva,Valeria Santos,190
3,C0008,E0008,C0008_M03,3,Martin Lopez,Martin Ramos,228
4,C0010,E0010,C0010_M03,3,Paula Gray,Paula Campos,48


In [4]:
# reconstruct defective conditions
SEVERITIES=[.10,.30,.60]

entity_conditions={"clean":clean_eval.copy()}
entity_audits={}

for severity in SEVERITIES:
    defective,audit=inject_entity_inconsistency(
        clean_eval,
        entity_plan,
        severity=severity
    )

    entity_conditions[severity]=defective
    entity_audits[severity]=audit

    print(
        f"{severity:.0%} | affected =",
        len(audit)
    )

10% | affected = 30
30% | affected = 90
60% | affected = 180


In [5]:
# compare affected IDs against the frozen artifact
saved_affected=pd.read_csv(
    DATA_DIR/"entity_inconsistency_affected_campaign_ids.csv"
)

for severity in SEVERITIES:
    reconstructed=set(
        entity_audits[severity]["campaign_id"].astype(str)
    )

    frozen=set(
        saved_affected.loc[
            np.isclose(saved_affected["severity"],severity),
            "campaign_id"
        ].astype(str)
    )

    print(
        f"{severity:.0%} | "
        f"reconstructed={len(reconstructed)} | "
        f"frozen={len(frozen)} | "
        f"exact match={reconstructed==frozen}"
    )

    assert reconstructed==frozen

10% | reconstructed=30 | frozen=30 | exact match=True
30% | reconstructed=90 | frozen=90 | exact match=True
60% | reconstructed=180 | frozen=180 | exact match=True


In [6]:
# validate against official ones
entity_serialized={
    key:serialize_campaign_dataset(df)
    for key,df in entity_conditions.items()
}

for key,df in entity_serialized.items():
    print(key,df.shape)

clean (300, 2)
0.1 (300, 2)
0.3 (300, 2)
0.6 (300, 2)


In [7]:
# load minilm version used originaly
MINILM_MODEL_NAME="sentence-transformers/all-MiniLM-L6-v2"
MINILM_REVISION="1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

minilm_model=SentenceTransformer(
    MINILM_MODEL_NAME,
    revision=MINILM_REVISION
)

print(
    minilm_model.get_sentence_embedding_dimension(),
    minilm_model.max_seq_length
)

384 256


In [8]:
# regenerate embeddings temporarily
reconstructed_embeddings={}

for condition in ["clean",.10,.30,.60]:
    X,runtime=encode_serialized_campaigns(
        entity_serialized[condition],
        minilm_model,
        batch_size=64
    )

    reconstructed_embeddings[condition]=X

    print(
        condition,
        X.shape,
        f"{runtime:.2f}s"
    )

Token indices sequence length is longer than the specified maximum sequence length for this model (259 > 256). Running this sequence through the model will result in indexing errors


Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

clean (300, 384) 92.86s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.1 (300, 384) 97.65s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.3 (300, 384) 88.56s
Campaigns: 300
Chunks to encode: 308


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

0.6 (300, 384) 127.86s


In [9]:
# load formal .npy
official_embeddings={
    "clean":np.load(
        DATA_DIR/"entity_embeddings_clean.npy"
    ),
    .10:np.load(
        DATA_DIR/"entity_embeddings_10pct.npy"
    ),
    .30:np.load(
        DATA_DIR/"entity_embeddings_30pct.npy"
    ),
    .60:np.load(
        DATA_DIR/"entity_embeddings_60pct.npy"
    )
}

for key,X in official_embeddings.items():
    print(key,X.shape)

clean (300, 384)
0.1 (300, 384)
0.3 (300, 384)
0.6 (300, 384)


In [10]:
# compare recontrustec vs official
comparison=[]

for condition in ["clean",.10,.30,.60]:
    A=reconstructed_embeddings[condition]
    B=official_embeddings[condition]

    max_abs_diff=float(
        np.max(np.abs(A-B))
    )

    mean_abs_diff=float(
        np.mean(np.abs(A-B))
    )

    allclose=bool(
        np.allclose(
            A,
            B,
            atol=1e-6,
            rtol=1e-5
        )
    )

    comparison.append({
        "condition":condition,
        "shape_match":A.shape==B.shape,
        "max_abs_diff":max_abs_diff,
        "mean_abs_diff":mean_abs_diff,
        "allclose":allclose
    })

comparison_df=pd.DataFrame(comparison)

display(comparison_df)

,condition,shape_match,max_abs_diff,mean_abs_diff,allclose
0,clean,True,0.0,0.0,True
1,0.1,True,0.0,0.0,True
2,0.3,True,0.0,0.0,True
3,0.6,True,0.0,0.0,True


In [11]:
# save 
RECOVERY_DIR=DATA_DIR/"formal_defect_plans"
RECOVERY_DIR.mkdir(parents=True,exist_ok=True)

entity_plan.to_csv(
    RECOVERY_DIR/"entity_inconsistency_formal_plan_recovered.csv",
    index=False
)

print("Recovered formal plan saved.")

Recovered formal plan saved.


Freeze BGE

In [12]:
from huggingface_hub import model_info
from sentence_transformers import SentenceTransformer

BGE_MODEL_NAME="BAAI/bge-base-en-v1.5"
BGE_MODEL_REVISION=model_info(BGE_MODEL_NAME).sha
BGE_BATCH_SIZE=32

bge_model=SentenceTransformer(
    BGE_MODEL_NAME,
    revision=BGE_MODEL_REVISION
)

print("Model:",BGE_MODEL_NAME)
print("Revision:",BGE_MODEL_REVISION)
print("Dimension:",bge_model.get_sentence_embedding_dimension())
print("Max sequence length:",bge_model.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: BAAI/bge-base-en-v1.5
Revision: a5beb1e3e68b9ab74eb54cfd186867f64f240e1a
Dimension: 768
Max sequence length: 512


In [13]:
# validate chunking under BGE
from src.representations import split_campaign_into_chunks

chunk_records=[]

for condition in ["clean",.10,.30,.60]:
    for _,row in entity_serialized[condition].iterrows():

        chunks=split_campaign_into_chunks(
            row["serialized_text"],
            bge_model
        )

        lengths=[
            len(
                bge_model.tokenizer(
                    chunk,
                    add_special_tokens=True,
                    truncation=False
                )["input_ids"]
            )
            for chunk in chunks
        ]

        chunk_records.append({
            "condition":condition,
            "campaign_id":row["campaign_id"],
            "n_chunks":len(chunks),
            "max_chunk_tokens":max(lengths)
        })

bge_chunk_validation=pd.DataFrame(chunk_records)

display(
    bge_chunk_validation
    .groupby("condition")
    .agg(
        campaigns=("campaign_id","size"),
        multi_chunk=("n_chunks",lambda x:(x>1).sum()),
        max_chunks=("n_chunks","max"),
        max_chunk_tokens=("max_chunk_tokens","max")
    )
    .reset_index()
)

assert (
    bge_chunk_validation["max_chunk_tokens"]
    <=bge_model.max_seq_length
).all()

print("BGE chunk validation passed.")

,condition,campaigns,multi_chunk,max_chunks,max_chunk_tokens
0,0.1,300,0,1,261
1,0.3,300,0,1,261
2,0.6,300,0,1,261
3,clean,300,0,1,261


BGE chunk validation passed.


In [14]:
# generate the embeddings
from src.representations import encode_serialized_campaigns

entity_bge={}
entity_bge_runtime={}

for condition in ["clean",.10,.30,.60]:

    X,runtime=encode_serialized_campaigns(
        entity_serialized[condition],
        bge_model,
        batch_size=BGE_BATCH_SIZE
    )

    entity_bge[condition]=X
    entity_bge_runtime[condition]=runtime

    print(
        condition,
        X.shape,
        f"{runtime:.2f}s"
    )

Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

clean (300, 768) 269.18s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.1 (300, 768) 216.64s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.3 (300, 768) 200.75s
Campaigns: 300
Chunks to encode: 300


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

0.6 (300, 768) 208.80s


In [15]:
# validation 
for condition,X in entity_bge.items():

    assert X.shape[0]==300
    assert X.shape[1]==bge_model.get_sentence_embedding_dimension()
    assert np.isfinite(X).all()

    norms=np.linalg.norm(X,axis=1)

    assert np.allclose(
        norms,
        1.0,
        atol=1e-5
    )

print("All BGE Entity embeddings passed validation.")

All BGE Entity embeddings passed validation.


In [16]:
# verify only affected campaigns change under bge
for severity in [.10,.30,.60]:

    distances=np.linalg.norm(
        entity_bge[severity]
        -entity_bge["clean"],
        axis=1
    )

    changed_ids=set(
        entity_serialized["clean"].loc[
            distances>1e-8,
            "campaign_id"
        ]
    )

    expected_ids=set(
        entity_audits[severity]["campaign_id"]
    )

    print(
        f"{severity:.0%} | "
        f"changed={len(changed_ids)} | "
        f"expected={len(expected_ids)} | "
        f"exact match={changed_ids==expected_ids}"
    )

    assert changed_ids==expected_ids

10% | changed=30 | expected=30 | exact match=True
30% | changed=90 | expected=90 | exact match=True
60% | changed=180 | expected=180 | exact match=True


In [17]:
# save the primary artifacts 
BGE_DIR=DATA_DIR/"robustness_bge"
BGE_DIR.mkdir(parents=True,exist_ok=True)

names={
    "clean":"entity_bge_clean.npy",
    .10:"entity_bge_10pct.npy",
    .30:"entity_bge_30pct.npy",
    .60:"entity_bge_60pct.npy"
}

for condition,name in names.items():
    np.save(
        BGE_DIR/name,
        entity_bge[condition]
    )

entity_serialized["clean"][["campaign_id"]].to_csv(
    BGE_DIR/"entity_bge_campaign_ids.csv",
    index=False
)

print("Saved to:",BGE_DIR)

Saved to: /Users/jessica/fidelity-metric-lab/data/processed/robustness_bge


In [18]:
# save config 
import json

bge_config={
    "scenario":"Entity Inconsistency",
    "model":BGE_MODEL_NAME,
    "revision":BGE_MODEL_REVISION,
    "embedding_dimension":int(
        bge_model.get_sentence_embedding_dimension()
    ),
    "normalized":True,
    "batch_size":BGE_BATCH_SIZE,
    "campaign_serialization":"same as formal MiniLM experiment",
    "chunking":"same message-boundary procedure as formal MiniLM experiment",
    "aggregation":"token-weighted chunk mean followed by L2 normalization",
    "source_conditions_recovered_exactly":True,
    "minilm_reconstruction_max_abs_diff":0.0
}

with open(
    BGE_DIR/"entity_bge_configuration.json",
    "w"
) as f:
    json.dump(bge_config,f,indent=4)

print("Configuration saved.")

Configuration saved.
